In [1]:
import os
import subprocess
import json
from pathlib import Path

# 저장 경로
BASE_DIR = Path(r"C:\Users\user\Desktop\deepfake-detector")
VIDEO_DIR = BASE_DIR / "data" / "real_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {VIDEO_DIR}")

저장 경로: C:\Users\user\Desktop\deepfake-detector\data\real_videos


In [2]:
# 내셔널지오그래픽 계열 쿼리 (70개 목표)
nat_geo_queries = [
    "national geographic documentary wildlife 2018",
    "national geographic documentary ocean 2017",
    "national geographic documentary animals 2019",
    "national geographic full documentary 2016",
    "BBC nature documentary 2018",
    "BBC earth documentary 2017",
    "discovery channel documentary 2018",
]

# 브이로그 쿼리 (30개 목표)
vlog_queries = [
    "daily vlog 2018 travel",
    "daily vlog 2017 lifestyle",
    "travel vlog 2019",
    "vlog 2016 daily life",
]

print(f"다큐 쿼리 수: {len(nat_geo_queries)}")
print(f"브이로그 쿼리 수: {len(vlog_queries)}")

다큐 쿼리 수: 7
브이로그 쿼리 수: 4


In [3]:
def collect_urls(queries, max_per_query=15, category="unknown"):
    results = []
    for query in queries:
        print(f"\n[검색 중] {query}")
        cmd = [
            "yt-dlp",
            f"ytsearch{max_per_query}:{query}",
            "--dump-json",
            "--no-download",
            "--quiet",
        ]
        try:
            output = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            for line in output.stdout.strip().split("\n"):
                if not line:
                    continue
                try:
                    info = json.loads(line)
                    results.append({
                        "id": info.get("id"),
                        "title": info.get("title"),
                        "upload_date": info.get("upload_date"),  # YYYYMMDD 형식
                        "duration": info.get("duration"),        # 초 단위
                        "url": f"https://www.youtube.com/watch?v={info.get('id')}",
                        "category": category,
                    })
                except json.JSONDecodeError:
                    continue
            print(f"  → {len([r for r in results if r['category']==category])}개 수집됨")
        except subprocess.TimeoutExpired:
            print(f"  → 타임아웃, 다음 쿼리로")
            continue
    return results

# 수집 실행
print("=== 다큐멘터리 URL 수집 ===")
nat_geo_results = collect_urls(nat_geo_queries, max_per_query=15, category="documentary")

print("\n=== 브이로그 URL 수집 ===")
vlog_results = collect_urls(vlog_queries, max_per_query=15, category="vlog")

all_results = nat_geo_results + vlog_results
print(f"\n총 수집: {len(all_results)}개")

=== 다큐멘터리 URL 수집 ===

[검색 중] national geographic documentary wildlife 2018
  → 15개 수집됨

[검색 중] national geographic documentary ocean 2017
  → 30개 수집됨

[검색 중] national geographic documentary animals 2019
  → 45개 수집됨

[검색 중] national geographic full documentary 2016
  → 60개 수집됨

[검색 중] BBC nature documentary 2018
  → 75개 수집됨

[검색 중] BBC earth documentary 2017
  → 90개 수집됨

[검색 중] discovery channel documentary 2018
  → 105개 수집됨

=== 브이로그 URL 수집 ===

[검색 중] daily vlog 2018 travel
  → 14개 수집됨

[검색 중] daily vlog 2017 lifestyle
  → 29개 수집됨

[검색 중] travel vlog 2019
  → 44개 수집됨

[검색 중] vlog 2016 daily life
  → 59개 수집됨

총 수집: 164개


In [5]:
def filter_videos(results, before_year=2020, min_duration=60, max_duration=3600):
    seen_ids = set()
    filtered = []
    
    for v in results:
        vid_id = v["id"]
        date = v["upload_date"]  # YYYYMMDD
        duration = v["duration"] or 0
        
        # 중복 제거
        if vid_id in seen_ids:
            continue
        seen_ids.add(vid_id)
        
        # 날짜 필터 (2020년 이전)
        if date and int(date[:4]) >= before_year:
            continue
        
        # 길이 필터 (1분~60분)
        if not (min_duration <= duration <= max_duration):
            continue
        
        filtered.append(v)
    
    return filtered

filtered = filter_videos(all_results)

# 카테고리별 확인
docs = [v for v in filtered if v["category"] == "documentary"]
vlogs = [v for v in filtered if v["category"] == "vlog"]

print(f"필터링 후 총: {len(filtered)}개")
print(f"  다큐: {docs}개")
print(f"  브이로그: {vlogs}개")

# 목록 출력
for v in filtered[:10]:
    print(f"  [{v['upload_date']}] {v['title'][:50]} ({v['duration']//60}분)")

필터링 후 총: 84개
  다큐: [{'id': 'X0tkj1pAiuk', 'title': 'National Geographic Documentary - The Greatest Apex Predators on Earth - New Documentary HD 2018', 'upload_date': '20171130', 'duration': 2260, 'url': 'https://www.youtube.com/watch?v=X0tkj1pAiuk', 'category': 'documentary'}, {'id': 'Onlrg0IX2aE', 'title': 'Animal Attack - National Geographic - [ Documentary ]', 'upload_date': '20150722', 'duration': 2475, 'url': 'https://www.youtube.com/watch?v=Onlrg0IX2aE', 'category': 'documentary'}, {'id': 'bZyz_tvYPBM', 'title': 'Wildlife Animal National Geographic Documentary 2019', 'upload_date': '20190811', 'duration': 2707, 'url': 'https://www.youtube.com/watch?v=bZyz_tvYPBM', 'category': 'documentary'}, {'id': 'Fd95sDDARdA', 'title': 'National Geographic Documentary - Fighting to Survive Wild Nature - Wildlife Animal', 'upload_date': '20171021', 'duration': 2995, 'url': 'https://www.youtube.com/watch?v=Fd95sDDARdA', 'category': 'documentary'}, {'id': 'e6QBg7Vbt_c', 'title': 'National Geograp

In [6]:
import time

def download_videos(video_list, save_dir, target_count, label):
    save_dir = Path(save_dir) / label
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 이미 받은 파일 확인
    existing = set(p.stem for p in save_dir.glob("*.mp4"))
    print(f"[{label}] 이미 받은 영상: {len(existing)}개")
    
    success = 0
    fail = 0
    
    for i, v in enumerate(video_list):
        if success >= target_count:
            print(f"\n목표 {target_count}개 달성!")
            break
        
        vid_id = v["id"]
        if vid_id in existing:
            print(f"[{i+1}] 스킵 (이미 존재): {vid_id}")
            success += 1
            continue
        
        print(f"\n[{i+1}/{len(video_list)}] 다운로드 중: {v['title'][:50]}")
        print(f"  날짜: {v['upload_date']} | 길이: {v['duration']//60}분")
        
        cmd = [
            "yt-dlp",
            v["url"],
            "-o", str(save_dir / "%(id)s.%(ext)s"),
            "--format", "mp4/bestvideo[height<=720]+bestaudio/best[height<=720]",
            "--write-info-json",   # 메타데이터 저장
            "--no-playlist",
            "--quiet",
            "--progress",
        ]
        
        try:
            result = subprocess.run(cmd, timeout=300)
            if result.returncode == 0:
                success += 1
                print(f"  ✓ 완료 ({success}개 성공)")
            else:
                fail += 1
                print(f"  ✗ 실패")
        except subprocess.TimeoutExpired:
            fail += 1
            print(f"  ✗ 타임아웃")
        
        time.sleep(1)  # 서버 부하 방지
    
    print(f"\n[{label}] 최종: 성공 {success}개 / 실패 {fail}개")
    return success

# 다운로드 실행
print("=== 다큐멘터리 다운로드 ===")
doc_count = download_videos(docs, VIDEO_DIR, target_count=70, label="documentary")

print("\n=== 브이로그 다운로드 ===")
vlog_count = download_videos(vlogs, VIDEO_DIR, target_count=30, label="vlog")

print(f"\n최종 합계: {doc_count + vlog_count}개")

=== 다큐멘터리 다운로드 ===
[documentary] 이미 받은 영상: 51개
[1] 스킵 (이미 존재): X0tkj1pAiuk
[2] 스킵 (이미 존재): Onlrg0IX2aE
[3] 스킵 (이미 존재): bZyz_tvYPBM
[4] 스킵 (이미 존재): Fd95sDDARdA

[5/46] 다운로드 중: National Geographic Documentary Komodo Dragon isla
  날짜: 20180722 | 길이: 34분
  ✓ 완료 (5개 성공)
[6] 스킵 (이미 존재): O-k4tcjwZK8

[7/46] 다운로드 중: National Geographic Documentary - Endangered Speci
  날짜: 20171129 | 길이: 51분
  ✓ 완료 (7개 성공)
[8] 스킵 (이미 존재): -0KrV5WsZq8
[9] 스킵 (이미 존재): bPeOe2wBYbs
[10] 스킵 (이미 존재): MfWyzrkFkg8
[11] 스킵 (이미 존재): Kmb2Anx2QDg
[12] 스킵 (이미 존재): ZiULxLLP32s
[13] 스킵 (이미 존재): B2CyvXd9gx4
[14] 스킵 (이미 존재): nqGM-FJMJWA
[15] 스킵 (이미 존재): vdMmKXKgPoM
[16] 스킵 (이미 존재): qg5ucav7hr4
[17] 스킵 (이미 존재): XTECpZ0UW-w
[18] 스킵 (이미 존재): qyTgt6NysKE
[19] 스킵 (이미 존재): PBSB3kysCSk
[20] 스킵 (이미 존재): MOrC7h43Hko
[21] 스킵 (이미 존재): RAm5nGtUZiQ
[22] 스킵 (이미 존재): 6ucJJIeAFr8

[23/46] 다운로드 중: Antarctica - National Geographic Explorer - Nov 29
  날짜: 20161213 | 길이: 33분
  ✓ 완료 (23개 성공)
[24] 스킵 (이미 존재): eleScfXwwM0
[25] 스킵 (이미 존재): 2Co_hmLenD8

In [7]:
from pathlib import Path

VIDEO_DIR = Path(r"C:\Users\user\Desktop\deepfake-detector\data\real_videos")

doc_videos = list((VIDEO_DIR / "documentary").glob("*.mp4"))
vlog_videos = list((VIDEO_DIR / "vlog").glob("*.mp4"))

print(f"다큐: {len(doc_videos)}개")
print(f"브이로그: {len(vlog_videos)}개")
print(f"합계: {len(doc_videos) + len(vlog_videos)}개")

다큐: 56개
브이로그: 33개
합계: 89개
